In [ ]:
!git clone https://github.com/matiimonti/Transformer_project
%cd Transformer_project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git pull

## Chess Transformer

In [ ]:
import urllib.request
import os
import io
import sys
import importlib
sys.path.insert(0, 'src')

import pgn_data
importlib.reload(pgn_data)

# Decompressor for the .zip Lichess file
# !pip install zstandard -q
import zstandard as zstd

os.makedirs('data', exist_ok = True)

In [ ]:
# Import Lichess games 
YEAR = '2013'
MONTH = '01'
url = f'https://database.lichess.org/standard/lichess_db_standard_rated_{YEAR}-{MONTH}.pgn.zst'
zst_path = f'data/lichess_{YEAR}_{MONTH}.pgn.zst'
pgn_path = 'data/games.pgn'

print(f'Downloading: {url}')

### Stream download
def download(url, dest):
    def reporthook(count, block_size, total_size):
        mb_done = count * block_size / 1_000_000
        mb_total = total_size / 1_000_000 if total_size > 0 else '?'
        print(f'\r {mb_done:.1f} MB / {mb_total} MB', end='', flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=reporthook)
    print()

download(url, zst_path)
print(f"Compressed file saved: {zst_path}")


In [ ]:
### Decompress file
dctx = zstd.ZstdDecompressor()
with open(zst_path, 'rb') as compressed:
    with open(pgn_path, 'wb') as out_file:
        dctx.copy_stream(compressed, out_file)
print(f'File saved to: {pgn_path}')
print(f'File size: {os.path.getsize(pgn_path) / 1_000_000:.1f} MB')

In [ ]:
## Example check
with open(pgn_path) as f:
    sample = "".join([next(f) for _ in range(30)])

print("\nFirst 30 lines:")
print(sample)

### Parse Data 

In [ ]:
from pgn_data import parse_pgn, ChessTokenizer

SAMPLE_BYTES = 500_000

with open(pgn_path, 'r', encoding='utf-8', errors='ignore') as f:
    sample_text = f.read(SAMPLE_BYTES)

# Trim to last complete game (avoid cutting mid-game)
last_newline = sample_text.rfind('\n\n')
if last_newline != -1:
    sample_text = sample_text[:last_newline]

sample_games = parse_pgn(sample_text)
print(f'Games parsed from sample: {len(sample_games)}')
print(f'Avg moves per game:       {sum(len(g) for g in sample_games) / len(sample_games):.1f}')


In [ ]:
# Inspect the first 3 games
for i, game in enumerate(sample_games[:3]):
    print(f'--- Game {i+1} ({len(game)} moves) ---')
    print(' '.join(game))
    print()

In [ ]:
# Tokenizer
tok = ChessTokenizer()
tok.build_from_games(sample_games)

print(f"Vocab size: {tok.vocab_size}")

# Check on first game
encoded = tok.encode(sample_games[0])
decoded = tok.decode(encoded)

print(f"Original: {sample_games[0][:5]}")
print(f"Encoded:  {encoded[:7]}")
print(f"Decoded:  {decoded[:7]}")

# The key check — decoded middle should match original exactly
assert decoded[1:-1] == sample_games[0], "Round-trip failed!"
print("Round-trip passed.")